In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.impute import SimpleImputer

def add_player_comparisons(
    df,
    draft_year=2025,
    stat_columns=None,
    distance_metric='euclidean',
    include_combine=True,
    top_n=3
):
    
    # Default feature selection
    if stat_columns is None:
        stat_columns = [
            'pts', 'reb', 'ast', 'stl', 'blk', 'to',
            'fg%', '3p%', '3pa_rate', 'fta_rate', 'ft%',
            'effective_fg%', 'true_shooting_%', 'usg%',
            'ast/usg', 'ast/to', 'per', 'ortg', 'drtg',
            'obpm', 'dbpm', 'bpm',
            'draft_age', 'height', 'weight'
        ]
        
        if include_combine:
            combine_cols = [
                'wingspan', 'max_vertical', 'lane_agility', 
                'shuttle', '3/4_sprint', 'standing_reach'
            ]
            stat_columns.extend(combine_cols)
    
    # Filter columns that exist in the dataframe
    available_cols = [col for col in stat_columns if col in df.columns]
    
    # Separate target draft class from historical data
    target_players = df[df['draft_year'] == draft_year].copy()
    historical_players = df[df['draft_year'] != draft_year].copy()
    
    if len(target_players) == 0:
        raise ValueError(f"No players found for draft year {draft_year}")
    
    # Handle missing values - impute with median
    imputer = SimpleImputer(strategy='median')
    
    # Fit imputer on historical data
    historical_features = imputer.fit_transform(historical_players[available_cols])
    target_features = imputer.transform(target_players[available_cols])
    
    # Standardize features
    scaler = StandardScaler()
    historical_scaled = scaler.fit_transform(historical_features)
    target_scaled = scaler.transform(target_features)
    
    # Calculate distances
    if distance_metric == 'euclidean':
        distances = euclidean_distances(target_scaled, historical_scaled)
    elif distance_metric == 'manhattan':
        from sklearn.metrics.pairwise import manhattan_distances
        distances = manhattan_distances(target_scaled, historical_scaled)
    else:
        raise ValueError("distance_metric must be 'euclidean' or 'manhattan'")
    
    # Find the top N most similar players for each target player
    comparison_data = []
    
    for i, target_idx in enumerate(target_players.index):
        target_name = target_players.loc[target_idx, 'name']
        
        # Get indices of top N most similar players (minimum distances)
        top_n_indices = np.argsort(distances[i])[:top_n]
        
        comparison_row = {'name': target_name}
        
        for rank, similar_idx in enumerate(top_n_indices, 1):
            dist = distances[i][similar_idx]
            hist_player_idx = historical_players.index[similar_idx]
            similar_player_name = historical_players.loc[hist_player_idx, 'name']
            
            comparison_row[f'comparison_{rank}'] = similar_player_name
            comparison_row[f'distance_{rank}'] = dist
        
        comparison_data.append(comparison_row)
    
    # Create comparison dataframe
    comp_df = pd.DataFrame(comparison_data)
    
    # Normalize distances to 0-100 similarity scores for each comparison
    for rank in range(1, top_n + 1):
        distance_col = f'distance_{rank}'
        score_col = f'similarity_score_{rank}'
        
        # Get all distances for this rank
        all_distances = comp_df[distance_col].values
        max_distance = all_distances.max()
        min_distance = all_distances.min()
        
        if max_distance > min_distance:
            comp_df[score_col] = 100 - (
                (comp_df[distance_col] - min_distance) / 
                (max_distance - min_distance) * 100
            )
        else:
            comp_df[score_col] = 100.0
        
        # Round to 1 decimal place
        comp_df[score_col] = comp_df[score_col].round(1)
    
    # Drop distance columns (we don't need them in final output)
    distance_cols = [f'distance_{i}' for i in range(1, top_n + 1)]
    comp_df = comp_df.drop(columns=distance_cols)
    
    return comp_df


# Example usage:
# Load your data
df = pd.read_csv('NBADraftwithDARKO.csv')

# Add comparison columns for 2025 draft
df_comparisons = add_player_comparisons(
    df,
    draft_year=2025,
    distance_metric='euclidean',
    include_combine=True,  # Set to False if combine data is mostly missing
    top_n=3
)

# View the results
print(df_comparisons)

# Save the comparison data
df_comparisons.to_csv('2025_draft_comparisons.csv', index=False)

print("\nComparisons saved to '2025_draft_comparisons.csv'")
print(f"Total players analyzed: {len(df_comparisons)}")

                 name         comparison_1       comparison_2  \
0        Cooper Flagg          Otto Porter     Frank Kaminsky   
1        Dylan Harper       Markelle Fultz         Jaden Ivey   
2      V.J. Edgecombe          Gary Harris    Josh Richardson   
3        Kon Knueppel         Luke Kennard       Jared McCain   
4          Ace Bailey       Taurean Prince       Justin Lewis   
..                ...                  ...                ...   
78        Tamar Bates  Mantas Rubstavicius        Jeremy Lamb   
79    Payton Sandfort       Trey Alexander  Admiral Schofield   
80       Kobe Johnson          Dalen Terry      Cason Wallace   
81    Jaxson Robinson          Jett Howard       Jaylen Wells   
82  Matthew Cleveland       Antonio Reeves    Jerome Robinson   

           comparison_3  similarity_score_1  similarity_score_2  \
0   Sindarius Thornwell                38.6                36.7   
1        Reggie Jackson                80.9                80.0   
2          Dion Wa